# Automatic hyperparameter search with KerasTuner

A search space, a Bayesian tuner, and the two things that separate a usable result from a contaminated one.

**Runs on:** CPU — about 10 minutes &nbsp;·&nbsp; **Slides:** [Chapter 18 — Best Practices for the Real World](../../../course-web-slides/ch18/index.html) &nbsp;·&nbsp; **Section:** 01 — Hyperparameter optimization

---

## Install and set up

In [ ]:
# !pip install keras-tuner -q
import keras
from keras import layers
import keras_tuner as kt
import numpy as np

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape((-1, 28 * 28)).astype("float32") / 255
x_test = x_test.reshape((-1, 28 * 28)).astype("float32") / 255

x_train_full, y_train_full = x_train[:], y_train[:]
num_val = 10000
x_train, x_val = x_train[:-num_val], x_train[-num_val:]
y_train, y_val = y_train[:-num_val], y_train[-num_val:]
print(f"train {len(x_train)}  val {len(x_val)}  test {len(x_test)}")

## A search space is a model-building function

In [ ]:
def build_model(hp):
    units = hp.Int(name="units", min_value=16, max_value=64, step=16)
    model = keras.Sequential([
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
    model.compile(optimizer=optimizer,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

Hardcoded values become **ranges**. Four kinds exist: `Int`, `Float`, `Boolean`, `Choice`. After sampling they are ordinary Python constants — the function is called once per trial with concrete values.

## A wider space, to make the point

In [ ]:
def build_deeper(hp):
    model = keras.Sequential()
    for i in range(hp.Int("num_layers", 1, 3)):
        model.add(layers.Dense(
            hp.Int(f"units_{i}", 32, 256, step=32), activation="relu"))
        if hp.Boolean(f"dropout_{i}"):
            model.add(layers.Dropout(hp.Float(f"rate_{i}", 0.1, 0.5, step=0.1)))
    model.add(layers.Dense(10, activation="softmax"))
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Float("lr", 1e-4, 1e-2, sampling="log")),
        loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

print("The space grows combinatorially:")
print("  3 depths x 8 widths^3 x 2 dropout^3 x 5 rates^3 x learning rate")
print("  = millions of configurations, from six lines.")
print()
print("That is why designing the space is your job, not the tuner's.")

> **Note** — `sampling="log"` on the learning rate. Sampling it uniformly between 1e-4 and 1e-2 would put 90% of the trials above 1e-3 — **learning rates should be searched on a log scale**, always.

## The tuner

In [ ]:
tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=20,
    executions_per_trial=2,
    directory="mnist_kt_test",
    overwrite=True,
)
tuner.search_space_summary()

**`executions_per_trial=2`** is the answer to noisy feedback. Chapter 18 named the problem — *is 0.2% a better configuration or a lucky initialization?* — and this is the fix: train each configuration twice and average.

**`overwrite=False`** resumes a crashed search from the trial logs on disk. Set the directory somewhere durable before starting a multi-day run.

## Searching

In [ ]:
callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=5)]

tuner.search(
    x_train, y_train,
    batch_size=128,
    epochs=100,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=2,
)

> ⚠️ **Never pass the test set as `validation_data` here.** You would overfit to it immediately, and every number you report afterwards would be fiction.

`epochs=100` with `EarlyStopping` — you do not know in advance how many epochs each configuration needs, so give a generous budget and let the callback cut each run short.

## The results

In [ ]:
tuner.results_summary(num_trials=5)

best_hps = tuner.get_best_hyperparameters(4)
for i, hp in enumerate(best_hps):
    print(f"{i}: {hp.values}")

## Retraining properly

In [ ]:
def get_best_epoch(hp):
    model = build_model(hp)
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", mode="min",
                                        patience=10)]
    history = model.fit(x_train, y_train,
                        validation_data=(x_val, y_val),
                        epochs=100, batch_size=128, callbacks=cb, verbose=0)
    v = history.history["val_loss"]
    best = int(np.argmin(v)) + 1
    print(f"  best epoch: {best}")
    return best

def get_best_trained_model(hp):
    best_epoch = get_best_epoch(hp)
    model = build_model(hp)
    model.fit(x_train_full, y_train_full, batch_size=128,
              epochs=int(best_epoch * 1.2), verbose=0)
    return model

best_models = []
for i, hp in enumerate(best_hps):
    print(f"config {i}: {hp.values}")
    m = get_best_trained_model(hp)
    acc = m.evaluate(x_test, y_test, verbose=0)[1]
    print(f"  test accuracy: {acc:.4f}\n")
    best_models.append(m)

Two details, both easy to skip:

**A much higher patience** (10, not 5) in this second pass. The aggressive patience saved time during the search and may have left models underfitted.

**`* 1.2` epochs, and training on the full data** — validation folded back in, because there are no more hyperparameter decisions to make with it.

## The shortcut, and the warning

In [ ]:
quick = tuner.get_best_models(4)
print("reloaded from the search, without retraining:")
for i, m in enumerate(quick):
    print(f"  {i}: {m.evaluate(x_test, y_test, verbose=0)[1]:.4f}")
print()
print("Slightly worse than a proper retrain, and one line.")

> ⚠️ **Validation-set overfitting.** You have been updating hyperparameters using a signal computed on validation data — which means you have been **training them on it**, and they will overfit to it.

That is the entire purpose of keeping a separate test set, and it is why the test set is a **one-shot instrument**.

## Premade search spaces

In [ ]:
print("KerasTuner ships tunable versions of the Keras Applications:")
print("  kt.applications.HyperXception")
print("  kt.applications.HyperResNet")
print()
print("Add data, run the search, get a pretty good model. Worth knowing")
print("because the higher-level decisions -- 'should I use residual")
print("connections throughout?' -- generalize across tasks in a way that")
print("'how many units in layer 2' never does.")

---

## What to take away

- Replace constants with ranges; search learning rates on a **log scale**.
- `executions_per_trial` averages away the noise in the feedback signal.
- Retrain the winners with higher patience, on the full data, for ~20% more epochs.
- **Tuning trains hyperparameters on the validation set.** The test set is a one-shot instrument.